# 05 — Religion & Social Demographics
**Source:** India Districts Census 2011 · 640 districts  
Covers religious composition, diversity index, and state-level breakdown.

In [ ]:
import sqlite3, pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings; warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid'); plt.rcParams['figure.dpi'] = 130

DB = '../data/processed/census_model.db'
conn = sqlite3.connect(DB)

df = pd.read_sql("""
    SELECT d.state_name, d.district_name,
           f.population,
           f.hindus, f.muslims, f.christians,
           f.sikhs, f.buddhists, f.jains,
           f.others_religions, f.religion_not_stated
    FROM fact_census f
    JOIN dim_location d ON f.location_id = d.location_id
""", conn)
conn.close()

rel_cols = ['hindus','muslims','christians','sikhs','buddhists','jains','others_religions']
rel_labels = ['Hindus','Muslims','Christians','Sikhs','Buddhists','Jains','Others']

for col in rel_cols:
    df[col+'_pct'] = (df[col] / df['population'] * 100).round(2)

# Diversity index using Simpson's diversity index (1 - sum(p_i^2))
df['diversity_index'] = 1 - sum((df[c] / df['population'])**2 for c in rel_cols)
df['diversity_index'] = df['diversity_index'].round(4)

state = df.groupby('state_name').sum(numeric_only=True).reset_index()
for col in rel_cols:
    state[col+'_pct'] = (state[col] / state['population'] * 100).round(2)
state['diversity_index'] = (1 - sum((state[c] / state['population'])**2 for c in rel_cols)).round(4)

national = {label: df[col].sum() for col, label in zip(rel_cols, rel_labels)}
print('National religious composition:')
for k, v in national.items():
    print(f'  {k}: {v/df.population.sum()*100:.2f}%')

## 5.1 — National Religious Composition (Pie Chart)

In [ ]:
fig = px.pie(
    names=list(national.keys()),
    values=list(national.values()),
    title='National Religious Composition — India 2011 Census',
    color_discrete_sequence=px.colors.qualitative.Bold,
    hole=0.3
)
fig.update_traces(textinfo='percent+label')
fig.update_layout(height=480)
fig.show()

## 5.2 — Religious Composition by State (Stacked %)

In [ ]:
state_sorted = state.sort_values('hindus_pct', ascending=False)
colors_rel = px.colors.qualitative.Bold[:7]

fig = go.Figure()
for col, label, color in zip([c+'_pct' for c in rel_cols], rel_labels, colors_rel):
    fig.add_trace(go.Bar(name=label, x=state_sorted['state_name'], y=state_sorted[col], marker_color=color))

fig.update_layout(
    barmode='stack',
    title='Religious Composition by State (%) — sorted by Hindu %',
    yaxis_title='% of Population',
    xaxis_tickangle=-45,
    height=530,
    legend=dict(orientation='h', y=1.05)
)
fig.show()

## 5.3 — Muslim Population Share by State

In [ ]:
state_muslim = state.sort_values('muslims_pct', ascending=True)

fig, ax = plt.subplots(figsize=(10, 10))
ax.barh(state_muslim['state_name'], state_muslim['muslims_pct'],
        color=sns.color_palette('Greens_r', len(state_muslim)))
avg_muslim = df['muslims'].sum() / df['population'].sum() * 100
ax.axvline(avg_muslim, color='red', linestyle='--', label=f'National avg: {avg_muslim:.1f}%')
ax.set_xlabel('Muslim % of Population')
ax.set_title('Muslim Population Share by State', fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout(); plt.show()

## 5.4 — Religious Diversity Index by State (Simpson's Index)

In [ ]:
state_div = state.sort_values('diversity_index', ascending=True)

fig, ax = plt.subplots(figsize=(10, 10))
colors = ['#e74c3c' if d < 0.1 else '#f39c12' if d < 0.3 else '#3498db' if d < 0.5 else '#2ecc71'
          for d in state_div['diversity_index']]
ax.barh(state_div['state_name'], state_div['diversity_index'], color=colors)
ax.set_xlabel("Simpson's Diversity Index (0 = homogeneous, 1 = maximally diverse)")
ax.set_title("Religious Diversity Index by State\n(Red very low | Orange low | Blue medium | Green high)",
             fontsize=12, fontweight='bold')
for i, (bar, val) in enumerate(zip(ax.patches, state_div['diversity_index'])):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=8)
plt.tight_layout(); plt.show()

## 5.5 — Christian, Sikh, Buddhist, Jain Share by State (Heatmap)

In [ ]:
minority_cols = ['christians_pct','sikhs_pct','buddhists_pct','jains_pct']
minority_labels = ['Christians %','Sikhs %','Buddhists %','Jains %']

heat = state.set_index('state_name')[minority_cols].copy()
heat.columns = minority_labels
heat = heat.sort_values('Christians %', ascending=False)

fig, ax = plt.subplots(figsize=(9, 12))
sns.heatmap(heat, annot=True, fmt='.1f', cmap='YlOrRd', linewidths=.4,
            cbar_kws={'label': '% of State Population'}, ax=ax)
ax.set_title('Minority Religion Share by State (%)', fontsize=12, fontweight='bold')
ax.set_xlabel('')
plt.tight_layout(); plt.show()

## 5.6 — Top 10 Most Religiously Diverse Districts

In [ ]:
top_diverse = df.nlargest(10, 'diversity_index')[['district_name','state_name','diversity_index'] + [c+'_pct' for c in rel_cols]]

fig = go.Figure()
for col, label, color in zip([c+'_pct' for c in rel_cols], rel_labels, colors_rel):
    fig.add_trace(go.Bar(
        name=label,
        x=top_diverse['district_name'] + ', ' + top_diverse['state_name'],
        y=top_diverse[col],
        marker_color=color
    ))

fig.update_layout(
    barmode='stack',
    title='Top 10 Most Religiously Diverse Districts — Religious Composition',
    yaxis_title='% of Population',
    xaxis_tickangle=-30,
    height=500,
    legend=dict(orientation='h', y=1.05)
)
fig.show()